# APIM ❤️ MCP from GraphQL

## Convert a GraphQL API into governed MCP tools

![Architecture](../../images/mcp-from-graphql.png)

This lab imports the public Countries GraphQL API into Azure API Management, uses [graphql-mcp](https://graphql-mcp.com/) to generate read-only MCP tools, exposes the remote MCP server through API Management, and registers an API Center integration that continuously discovers the GraphQL and MCP APIs from APIM. The generated tools are tested with Microsoft Agent Framework.

### Prerequisites

- Python 3.12
- VS Code with the Jupyter extension
- [uv](https://docs.astral.sh/uv/)
- Azure CLI signed into an Azure subscription
- Contributor plus RBAC Administrator, or Owner, on the subscription
- No local Docker installation is required; Azure Container Registry builds the image remotely.

▶️ Click `Run All` to execute all steps sequentially, or execute them `Step by Step`.

The deployment creates billable Azure resources.

### 0️⃣ Configure the lab environment

Install the notebook-only dependencies. Infrastructure dependencies for the translator are pinned separately in `src/graphql-mcp/requirements.txt`.

In [ ]:
%uv pip install "mcp==1.21.2" "agent-framework==1.7" requests

### 1️⃣ Import required libraries

Load the shared lab utilities and verify the Python runtime before creating resources.

In [ ]:
import asyncio
import json
import os
import sys
from pprint import pprint

import requests

sys.path.insert(1, "../../shared")
import utils

assert sys.version_info[:2] == (3, 12), "This lab requires Python 3.12."
utils.print_ok(f"Python {sys.version.split()[0]} is ready")

### 2️⃣ Initialize notebook variables

Configure deterministic deployment settings and the public Countries GraphQL endpoint. `graphql-mcp` requires introspection to be enabled on the source endpoint.

In [ ]:
deployment_name = os.path.basename(os.path.dirname(globals()["__vsc_ipynb_file__"]))
resource_group_name = f"lab-{deployment_name}"
resource_group_location = "swedencentral"

graphql_upstream_url = "https://countries.trevorblades.com/graphql"
graphql_api_path = "countries-graphql"
mcp_api_path = "countries-mcp"
inference_api_path = "inference"
foundry_project_name = deployment_name

apim_sku = "Basicv2"
apim_subscriptions_config = [
    {"name": "subscription1", "displayName": "Subscription 1"}
]
ai_services_config = [
    {"name": "foundry1", "location": "swedencentral"}
]
models_config = [
    {
        "name": "gpt-4.1-mini",
        "publisher": "OpenAI",
        "version": "2025-04-14",
        "sku": "GlobalStandard",
        "capacity": 50,
    }
]
translator_source = "src/graphql-mcp"
translator_image = "graphql-mcp"
translator_version = "2.1.5"

configuration = {
    "resource_group": resource_group_name,
    "location": resource_group_location,
    "graphql_upstream": graphql_upstream_url,
    "model": models_config[0]["name"],
    "apim_sku": apim_sku,
}
pprint(configuration)

In [ ]:
account = utils.run(
    "az account show",
    "Retrieved the current Azure account",
    "Failed to retrieve the current Azure account",
)
assert account.success and account.json_data, "Sign in with az login before continuing."
subscription_id = account.json_data["id"]
utils.print_info(f"Subscription: {account.json_data['name']} ({subscription_id})")

### 3️⃣ Create deployment using 🦾 Bicep

The Bicep template deploys monitoring, API Management, a Foundry model, a native pass-through GraphQL API, Azure Container Registry, Azure Container Apps, a passthrough MCP API, and an API Center APIM integration. API Center uses its system-assigned identity and the API Management Service Reader Role to discover and synchronize the managed APIs.

In [ ]:
utils.create_resource_group(resource_group_name, resource_group_location)

bicep_parameters = {
    "$schema": "https://schema.management.azure.com/schemas/2019-04-01/deploymentParameters.json#",
    "contentVersion": "1.0.0.0",
    "parameters": {
        "apimSku": {"value": apim_sku},
        "apimSubscriptionsConfig": {"value": apim_subscriptions_config},
        "aiServicesConfig": {"value": ai_services_config},
        "modelsConfig": {"value": models_config},
        "inferenceAPIPath": {"value": inference_api_path},
        "foundryProjectName": {"value": foundry_project_name},
        "graphqlUpstreamUrl": {"value": graphql_upstream_url},
        "graphqlAPIPath": {"value": graphql_api_path},
        "mcpAPIPath": {"value": mcp_api_path},
    },
}

with open("params.json", "w", encoding="utf-8") as parameters_file:
    json.dump(bicep_parameters, parameters_file, indent=2)

deployment = utils.run(
    f"az deployment group create --name {deployment_name} "
    f"--resource-group {resource_group_name} --template-file main.bicep "
    "--parameters params.json",
    f"Deployment '{deployment_name}' succeeded",
    f"Deployment '{deployment_name}' failed",
)
assert deployment.success, "The Azure deployment failed. Review the deployment diagnostics above."

In [ ]:
deployment_details = utils.run(
    f"az deployment group show --name {deployment_name} --resource-group {resource_group_name}",
    f"Retrieved deployment '{deployment_name}'",
    f"Failed to retrieve deployment '{deployment_name}'",
)
assert deployment_details.success and deployment_details.json_data

apim_resource_gateway_url = utils.get_deployment_output(
    deployment_details, "apimResourceGatewayURL", "APIM gateway URL"
)
graphql_endpoint = utils.get_deployment_output(
    deployment_details, "graphqlEndpoint", "GraphQL endpoint"
)
mcp_endpoint = utils.get_deployment_output(
    deployment_details, "mcpEndpoint", "MCP endpoint"
)
apic_service_name = utils.get_deployment_output(
    deployment_details, "apicServiceName", "API Center service"
)
apic_api_source_name = utils.get_deployment_output(
    deployment_details, "apicApiSourceName", "API Center APIM source"
)
container_registry_name = utils.get_deployment_output(
    deployment_details, "containerRegistryName", "Container Registry"
)
container_app_name = utils.get_deployment_output(
    deployment_details, "containerAppName", "Container App"
)

apim_subscriptions = json.loads(
    utils.get_deployment_output(deployment_details, "apimSubscriptions").replace("'", '"')
)
api_key = apim_subscriptions[0]["key"]
utils.print_info(f"Subscription key: ****{api_key[-4:]}")

### 4️⃣ Build and deploy the GraphQL MCP translator 📦

Azure Container Registry builds the pinned Python 3.12 image. The Container App then replaces its placeholder image with the `graphql-mcp` translator.

In [ ]:
image_reference = f"{container_registry_name}.azurecr.io/{translator_image}:{translator_version}"

build = utils.run(
    f"az acr build --registry {container_registry_name} "
    f"--resource-group {resource_group_name} --image {translator_image}:{translator_version} "
    f"--file {translator_source}/Dockerfile {translator_source} --no-logs",
    "Built and pushed the graphql-mcp image",
    "Failed to build the graphql-mcp image",
)
assert build.success

update = utils.run(
    f"az containerapp update --name {container_app_name} "
    f"--resource-group {resource_group_name} --image {image_reference}",
    "Updated the translator Container App",
    "Failed to update the translator Container App",
)
assert update.success

### 5️⃣ Test the GraphQL API through API Management 🧪

Confirm the native APIM GraphQL endpoint works before testing the generated MCP surface.

In [ ]:
def execute_graphql(query: str, variables: dict | None = None) -> dict:
    response = requests.post(
        graphql_endpoint,
        json={"query": query, "variables": variables or {}},
        timeout=30,
    )
    response.raise_for_status()
    return response.json()

country_result = execute_graphql(
    """
    query Country($code: ID!) {
      country(code: $code) {
        code
        name
        capital
        currency
        languages { name }
      }
    }
    """,
    {"code": "PT"},
)
assert country_result["data"]["country"]["name"] == "Portugal"
pprint(country_result)

### 6️⃣ Discover and invoke generated MCP tools 🧪

The MCP client connects only to API Management. APIM authenticates the subscription and forwards Streamable HTTP messages to the translator.

In [ ]:
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client


async def inspect_mcp_server() -> tuple[list[str], object]:
    headers = {"api-key": api_key}
    async with streamablehttp_client(mcp_endpoint, headers=headers) as streams:
        read_stream, write_stream, _ = streams
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_list = await session.list_tools()
            tool_names = [tool.name for tool in tool_list.tools]
            result = await session.call_tool("country", {"code": "PT"})
            return tool_names, result


mcp_tool_names, mcp_country_result = await inspect_mcp_server()
assert "country" in mcp_tool_names
utils.print_ok(f"Generated MCP tools: {', '.join(mcp_tool_names)}")
pprint(mcp_country_result)

### 7️⃣ Review the deployed endpoints 🔎

Display the deployed request path as a compact inventory. The architecture image above provides the full topology.

In [ ]:
from IPython.display import Markdown, display

endpoint_rows = [
    ("GraphQL source", graphql_upstream_url),
    ("APIM GraphQL", graphql_endpoint),
    ("APIM MCP", mcp_endpoint),
    ("APIM inference", f"{apim_resource_gateway_url}/{inference_api_path}/openai/v1"),
    ("API Center", apic_service_name),
]
endpoint_table = "| Surface | Endpoint |\n|---|---|\n" + "\n".join(
    f"| {name} | `{value}` |" for name, value in endpoint_rows
)
display(Markdown(endpoint_table))

### 8️⃣ Run automated checks 🧪

Cover the normal country lookup, a boundary lookup for an unknown code, an invalid GraphQL field, deterministic repeated results, and the generated MCP tool inventory.

In [ ]:
country_query = "query Country($code: ID!) { country(code: $code) { code name } }"

normal_result = execute_graphql(country_query, {"code": "PT"})
boundary_result = execute_graphql(country_query, {"code": "ZZ"})
invalid_result = execute_graphql("{ fieldThatDoesNotExist }")
repeat_result = execute_graphql(country_query, {"code": "PT"})

assert normal_result["data"]["country"] == {"code": "PT", "name": "Portugal"}
assert boundary_result["data"]["country"] is None
assert invalid_result.get("errors"), "An invalid field should return GraphQL errors."
assert repeat_result == normal_result
assert {"country", "countries", "continent", "continents"}.issubset(mcp_tool_names)
utils.print_ok("GraphQL and MCP automated checks passed")

### 9️⃣ Run a Microsoft Agent Framework agent with MCP tools 🧪

The agent sends model requests to the APIM OpenAI v1-compatible endpoint and consumes the generated MCP tools through the APIM MCP endpoint. No backend URL is exposed to the agent.

In [ ]:
from agent_framework import MCPStreamableHTTPTool
from agent_framework.openai import OpenAIChatClient
from httpx import AsyncClient


async def run_countries_agent(prompt: str):
    async with AsyncClient(headers={"api-key": api_key}) as mcp_http_client:
        async with MCPStreamableHTTPTool(
            name="countries",
            url=mcp_endpoint,
            http_client=mcp_http_client,
            description="Read country, continent, and language data from the Countries GraphQL API.",
        ) as countries_tool:
            client = OpenAIChatClient(
                base_url=f"{apim_resource_gateway_url}/{inference_api_path}/openai/v1",
                model=models_config[0]["name"],
                api_key=api_key,
            )
            agent = client.as_agent(
                name="CountriesAgent",
                instructions=(
                    "Answer country and continent questions using the countries MCP tools. "
                    "Use tool results rather than prior knowledge and keep answers concise."
                ),
                tools=[countries_tool],
            )
            return await agent.run(prompt)


agent_result = await run_countries_agent(
    "What is the capital of Portugal, and which languages are spoken there? and which continent is AF?"
)
print(agent_result)

### 🔟 Verify API Center dynamic discovery 🔎

Confirm that the APIM integration uses managed identity, targets the production environment, imports specifications, and discovers both GraphQL and MCP APIs. Synchronization is asynchronous, so this step waits for the expected API kinds instead of assuming asset names.

In [ ]:
api_center_workspace_url = (
    f"https://management.azure.com/subscriptions/{subscription_id}"
    f"/resourceGroups/{resource_group_name}/providers/Microsoft.ApiCenter"
    f"/services/{apic_service_name}/workspaces/default"
)
api_source_url = (
    f"{api_center_workspace_url}/apiSources/{apic_api_source_name}"
    "?api-version=2024-06-01-preview"
)
api_center_assets_url = (
    f"{api_center_workspace_url}/apis?api-version=2024-06-01-preview"
)

api_source = utils.run(
    f'az rest --method get --url "{api_source_url}"',
    "Retrieved the API Center APIM integration",
    "Failed to retrieve the API Center APIM integration",
)
assert api_source.success and api_source.json_data
source_properties = api_source.json_data["properties"]
assert source_properties["targetLifecycleStage"] == "production"
assert source_properties["targetEnvironmentId"].endswith("/environments/production-apim")
assert source_properties["importSpecification"] == "always"
assert source_properties["azureApiManagementSource"]["resourceId"].endswith(
    f"/Microsoft.ApiManagement/service/{apim_resource_gateway_url.split('//')[1].split('.')[0]}"
)

async def wait_for_discovered_api_kinds(timeout_seconds: int = 300) -> dict[str, str]:
    deadline = asyncio.get_running_loop().time() + timeout_seconds
    while True:
        assets = utils.run(
            f'az rest --method get --url "{api_center_assets_url}"',
            "Retrieved discovered API Center assets",
            "Failed to retrieve discovered API Center assets",
        )
        assert assets.success and assets.json_data
        discovered_kinds = {
            asset["name"]: asset["properties"]["kind"].lower()
            for asset in assets.json_data["value"]
        }
        if {"graphql", "mcp"}.issubset(discovered_kinds.values()):
            return discovered_kinds
        if asyncio.get_running_loop().time() >= deadline:
            raise TimeoutError(
                f"API Center discovery did not find GraphQL and MCP APIs within {timeout_seconds} seconds. "
                f"Current kinds: {discovered_kinds}"
            )
        await asyncio.sleep(10)

asset_kinds = await wait_for_discovered_api_kinds()
utils.print_ok("API Center discovered GraphQL and MCP APIs from API Management")
pprint(asset_kinds)

### 🧩 Optional exercises

- Change the upstream GraphQL endpoint to another introspection-enabled API.
- Add APIM GraphQL validation policies to constrain query depth and size.
- Compare the self-hosted translator with the hosted GraphQL MCP Bridge.
- Add authentication between APIM and a private GraphQL backend.

### 🗑️ Clean up resources

When finished, run [clean-up-resources.ipynb](clean-up-resources.ipynb) to delete the resource group and avoid ongoing charges.